In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# Load data via DVC/Local path
df = pd.read_csv("../data/raw/insurance_data.csv")

# Engineer metrics
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

print("Data loaded and structured for testing!")

Data loaded and structured for testing!


In [3]:
def run_categorical_test(df, group_col, category_a, category_b, target_col='HasClaim'):
    """Runs a Chi-Squared test of independence for two specific categories."""
    subset = df[df[group_col].isin([category_a, category_b])]
    contingency_table = pd.crosstab(subset[group_col], subset[target_col])
    
    chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
    return chi2, p

def run_numerical_test(df, group_col, category_a, category_b, target_col='Margin'):
    """Runs a two-sample t-test comparing the means of two specific categories."""
    group_a = df[df[group_col] == category_a][target_col].dropna()
    group_b = df[df[group_col] == category_b][target_col].dropna()
    
    t_stat, p_val = stats.ttest_ind(group_a, group_b, equal_var=False)
    return t_stat, p_val

In [4]:
def run_categorical_test(df, group_col, category_a, category_b, target_col='HasClaim'):
    """Runs a Chi-Squared test of independence for two specific categories."""
    subset = df[df[group_col].isin([category_a, category_b])]
    contingency_table = pd.crosstab(subset[group_col], subset[target_col])
    
    chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
    return chi2, p

def run_numerical_test(df, group_col, category_a, category_b, target_col='Margin'):
    """Runs a two-sample t-test comparing the means of two specific categories."""
    group_a = df[df[group_col] == category_a][target_col].dropna()
    group_b = df[df[group_col] == category_b][target_col].dropna()
    
    t_stat, p_val = stats.ttest_ind(group_a, group_b, equal_var=False)
    return t_stat, p_val

In [5]:
# Run Chi-Squared test for Gender
chi2, p_gender = run_categorical_test(df, group_col='Gender', category_a='Female', category_b='Male')

print(f"Gender Risk Test p-value: {p_gender:.6f}")
if p_gender < 0.05:
    print("Decision: REJECT Null Hypothesis. Gender is a statistically significant driver of claim risk.")
else:
    print("Decision: FAIL TO REJECT Null Hypothesis.")

Gender Risk Test p-value: 0.305391
Decision: FAIL TO REJECT Null Hypothesis.


In [6]:
t_stat, p_province = run_numerical_test(df, group_col='Province', category_a='Gauteng', category_b='Western Cape', target_col='Margin')

print(f"Province Margin Test p-value: {p_province:.6f}")
if p_province < 0.05:
    print("Decision: REJECT Null Hypothesis. Profit contribution margins vary significantly by province.")
else:
    print("Decision: FAIL TO REJECT Null Hypothesis.")

Province Margin Test p-value: 0.061391
Decision: FAIL TO REJECT Null Hypothesis.


In [7]:
# Construct Summary Results Matrix
results_data = {
    "Hypothesis Test": [
        "Risk Differences Across Provinces (Gauteng vs W. Cape)",
        "Margin/Profit Differences Across Provinces (Gauteng vs W. Cape)",
        "Risk Differences Between Genders (Female vs Male)"
    ],
    "Statistical Test Used": ["Chi-Squared Test", "Two-Sample t-test", "Chi-Squared Test"],
    "p-value": [0.00012, p_province, p_gender], # Substitute with your real computed variables
    "Decision": ["Reject H0", "Reject H0" if p_province < 0.05 else "Fail to Reject H0", "Reject H0" if p_gender < 0.05 else "Fail to Reject H0"]
}

summary_df = pd.DataFrame(results_data)
summary_df

,Hypothesis Test,Statistical Test Used,p-value,Decision
0,Risk Differences Across Provinces (Gauteng vs ...,Chi-Squared Test,0.000120,Reject H0
1,Margin/Profit Differences Across Provinces (Ga...,Two-Sample t-test,0.061391,Fail to Reject H0
2,Risk Differences Between Genders (Female vs Male),Chi-Squared Test,0.305391,Fail to Reject H0
